# Analyzing CIA Factbook Data Using SQL

In [1]:
%%capture
%load_ext sql
%sql sqlite:///factbook.db

## Introduction

In this project, we'll work with data from the CIA World Factbook (https://www.cia.gov/the-world-factbook/), a compendium of statistics about all of the countries on Earth. The Factbook contains demographic information like the following:

population — the global population.
population_growth — the annual population growth rate, as a percentage.
area — the total land and water area.

In this guided project, we'll use SQL in Jupyter Notebook to analyze data from this database.

## Overview of the Data

In [2]:
%%sql
SELECT *
FROM sqlite_master
WHERE type='table';

 * sqlite:///factbook.db
Done.


type,name,tbl_name,rootpage,sql
table,sqlite_sequence,sqlite_sequence,3,"CREATE TABLE sqlite_sequence(name,seq)"
table,facts,facts,47,"CREATE TABLE ""facts"" (""id"" INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL, ""code"" varchar(255) NOT NULL, ""name"" varchar(255) NOT NULL, ""area"" integer, ""area_land"" integer, ""area_water"" integer, ""population"" integer, ""population_growth"" float, ""birth_rate"" float, ""death_rate"" float, ""migration_rate"" float)"


In [3]:
%%sql
SELECT *
FROM facts
LIMIT 5;

 * sqlite:///factbook.db
Done.


id,code,name,area,area_land,area_water,population,population_growth,birth_rate,death_rate,migration_rate
1,af,Afghanistan,652230,652230,0,32564342,2.32,38.57,13.89,1.51
2,al,Albania,28748,27398,1350,3029278,0.3,12.92,6.58,3.3
3,ag,Algeria,2381741,2381741,0,39542166,1.84,23.67,4.31,0.92
4,an,Andorra,468,468,0,85580,0.12,8.13,6.96,0.0
5,ao,Angola,1246700,1246700,0,19625353,2.78,38.78,11.49,0.46


Here are the descriptions for some of the columns:

name — the name of the country.
area— the country's total area (both land and water).
area_land — the country's land area in square kilometers.
area_water — the country's water area in square kilometers.
population — the country's population.
population_growth— the country's population growth as a percentage.
birth_rate — the country's birth rate, or the number of births per year per 1,000 people.
death_rate — the country's death rate, or the number of death per year per 1,000 people.

## Summary Statistics

In [4]:
%%sql
SELECT MIN(population) AS min_pop,
       MAX(population) AS max_pop,
       MIN(population_growth) AS min_pop_growth,
       MAX(population_growth) AS max_pop_growth
FROM facts;

 * sqlite:///factbook.db
Done.


min_pop,max_pop,min_pop_growth,max_pop_growth
0,7256490011,0.0,4.02


Exploring Outliers:
There's a country with a population of 0
There's a country with a population of 7256490011 (or more than 7.2 billion people)

In [5]:
%%sql
SELECT *
FROM facts
ORDER BY population
LIMIT 20;

 * sqlite:///factbook.db
Done.


id,code,name,area,area_land,area_water,population,population_growth,birth_rate,death_rate,migration_rate
198,at,Ashmore and Cartier Islands,5,5,0,None,None,None,None,None
201,cr,Coral Sea Islands,3,3,0,None,None,None,None,None
202,hm,Heard Island and McDonald Islands,412,412,0,None,None,None,None,None
208,ip,Clipperton Island,6,6,0,None,None,None,None,None
210,fs,French Southern and Antarctic Lands,None,None,None,None,None,None,None,None
222,bv,Bouvet Island,49,49,0,None,None,None,None,None
223,jn,Jan Mayen,377,377,0,None,None,None,None,None
228,io,British Indian Ocean Territory,54400,60,54340,None,None,None,None,None
240,sx,South Georgia and South Sandwich Islands,3903,3903,0,None,None,None,None,None
244,bq,Navassa Island,5,5,0,None,None,None,None,None


There are regions that have no population since they are areas that are not habitable.

In [6]:
%%sql
SELECT *
FROM facts
ORDER BY population DESC
LIMIT 5;

 * sqlite:///factbook.db
Done.


id,code,name,area,area_land,area_water,population,population_growth,birth_rate,death_rate,migration_rate
261,xx,World,None,None,None,7256490011,1.08,18.6,7.8,None
37,ch,China,9596960,9326410,270550,1367485388,0.45,12.49,7.53,0.44
77,in,India,3287263,2973193,314070,1251695584,1.22,19.55,7.32,0.04
197,ee,European Union,4324782,None,None,513949445,0.25,10.2,10.2,2.5
186,us,United States,9826675,9161966,664709,321368864,0.78,12.49,8.15,3.86


The first row contains the information for the whole world; therefore, we can remove or exclude this row for our analysis.


## Exploring Average Population and Area

In [7]:
%%sql
SELECT MIN(population) AS min_pop,
       MAX(population) AS max_pop,
       MIN(population_growth) AS min_pop_growth,
       MAX(population_growth) AS max_pop_growth
FROM facts
WHERE name <> 'World';

 * sqlite:///factbook.db
Done.


min_pop,max_pop,min_pop_growth,max_pop_growth
0,1367485388,0.0,4.02


In [8]:
%%sql
SELECT AVG(population) AS avg_pop,
       AVG(area) AS avg_area
FROM facts
WHERE name <> 'World';

 * sqlite:///factbook.db
Done.


avg_pop,avg_area
32242666.56846473,555093.546184739


The average population is around 32 million and the average area is 555 thousand square kilometers.

## Finding Densely Populated Countries

In [9]:
%%sql
SELECT *
FROM facts
WHERE population > (SELECT AVG(population)
                    FROM facts
                    WHERE name <> 'World'
                   )
AND area < (SELECT AVG(area)
            FROM facts
            WHERE name <> 'World'); 

 * sqlite:///factbook.db
Done.


id,code,name,area,area_land,area_water,population,population_growth,birth_rate,death_rate,migration_rate
14,bg,Bangladesh,148460,130170,18290,168957745,1.6,21.14,5.61,0.46
65,gm,Germany,357022,348672,8350,80854408,0.17,8.47,11.42,1.24
80,iz,Iraq,438317,437367,950,37056169,2.93,31.45,3.77,1.62
83,it,Italy,301340,294140,7200,61855120,0.27,8.74,10.19,4.1
85,ja,Japan,377915,364485,13430,126919659,0.16,7.93,9.51,0.0
91,ks,"Korea, South",99720,96920,2800,49115196,0.14,8.19,6.75,0.0
120,mo,Morocco,446550,446300,250,33322699,1.0,18.2,4.81,3.36
138,rp,Philippines,300000,298170,1830,100998376,1.61,24.27,6.11,2.09
139,pl,Poland,312685,304255,8430,38562189,0.09,9.74,10.19,0.46
163,sp,Spain,505370,498980,6390,48146134,0.89,9.64,9.04,8.31


These are the countries that are densely populated.

Which country has the most people? Which country has the highest growth rate?

In [12]:
%%sql
SELECT name, population
FROM facts
WHERE name <> 'World'
ORDER BY population DESC
LIMIT 1;

 * sqlite:///factbook.db
Done.


name,population
China,1367485388


In [13]:
%%sql
SELECT name, population_growth
FROM facts
WHERE name <> 'World'
ORDER BY population_growth DESC
LIMIT 1;

 * sqlite:///factbook.db
Done.


name,population_growth
South Sudan,4.02


Which countries have the highest ratios of water to land? Which countries have more water than land?

In [14]:
%%sql
SELECT name, CAST(area_water AS Float) / area_land AS water_to_land_ratio
FROM facts
WHERE name <> 'World'
ORDER BY water_to_land_ratio DESC
LIMIT 5;

 * sqlite:///factbook.db
Done.


name,water_to_land_ratio
British Indian Ocean Territory,905.6666666666666
Virgin Islands,4.520231213872832
Puerto Rico,0.5547914317925592
"Bahamas, The",0.3866133866133866
Guinea-Bissau,0.2846728307254623


In [15]:
%%sql
SELECT name, area_water, area_land
FROM facts
WHERE area_water > area_land 
AND name <> 'World';

 * sqlite:///factbook.db
Done.


name,area_water,area_land
British Indian Ocean Territory,54340,60
Virgin Islands,1564,346


Which countries will add the most people to their populations next year?

In [17]:
%%sql
SELECT name, population_growth * population AS population_increase
FROM facts
WHERE name <> 'World'
ORDER BY population_increase DESC
LIMIT 5;

 * sqlite:///factbook.db
Done.


name,population_increase
India,1527068612.48
China,615368424.6
Nigeria,444827037.20000005
Pakistan,290665336.62
Ethiopia,287456216.91


Which countries have a higher death rate than birth rate?

In [19]:
%%sql
SELECT name, birth_rate, death_rate
FROM facts
WHERE death_rate > birth_rate 
AND name <> 'World';

 * sqlite:///factbook.db
Done.


name,birth_rate,death_rate
Austria,9.41,9.42
Belarus,10.7,13.36
Bosnia and Herzegovina,8.87,9.75
Bulgaria,8.92,14.44
Croatia,9.45,12.18
Czech Republic,9.63,10.34
Estonia,10.51,12.4
Germany,8.47,11.42
Greece,8.66,11.09
Hungary,9.16,12.73


Which countries have the highest population/area ratio, and how does it compare to list of densely populated couentries?

In [21]:
%%sql
SELECT name, population / area AS population_area_ratio
FROM facts
WHERE name <> 'World'
ORDER BY population_area_ratio DESC
LIMIT 5;

 * sqlite:///factbook.db
Done.


name,population_area_ratio
Macau,21168
Monaco,15267
Singapore,8141
Hong Kong,6445
Gaza Strip,5191


In [22]:
%%sql
SELECT *
FROM facts
WHERE population > (SELECT AVG(population)
                    FROM facts
                    WHERE name <> 'World'
                   )
AND area < (SELECT AVG(area)
            FROM facts
            WHERE name <> 'World')
LIMIT 5; 

 * sqlite:///factbook.db
Done.


id,code,name,area,area_land,area_water,population,population_growth,birth_rate,death_rate,migration_rate
14,bg,Bangladesh,148460,130170,18290,168957745,1.6,21.14,5.61,0.46
65,gm,Germany,357022,348672,8350,80854408,0.17,8.47,11.42,1.24
80,iz,Iraq,438317,437367,950,37056169,2.93,31.45,3.77,1.62
83,it,Italy,301340,294140,7200,61855120,0.27,8.74,10.19,4.1
85,ja,Japan,377915,364485,13430,126919659,0.16,7.93,9.51,0.0


The second query filters countries based on being above the average population and below the average area, while the first query focuses on finding the countries highest population and area ratio.